# Baseline — Linear Regression


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


Load Data

In [6]:
X_train  = pd.read_csv('X_train_prepared.csv')
y_train  = pd.read_csv('y_train.csv').squeeze()
X_test   = pd.read_csv('X_test_prepared.csv')
test_ids = pd.read_csv('test_ids.csv')

Pipeline - preprocessing with scaling and encoding

In [26]:
category_cols = ['weekday_of_release', 'season_of_release', 'lunar_phase']
numerical_cols = [c for c in X_train.columns if c not in category_cols]

num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler',  StandardScaler())
])
cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer([
    ('num', num_transformer, numerical_cols),
    ('cat', cat_transformer, category_cols),

], remainder='drop')

pipeline = Pipeline([
    ('prep',  preprocessor),
    ('model', LinearRegression())
])



Model fit

In [32]:
X_trn, X_val, y_trn, y_val = train_test_split(
    X_train, y_train, test_size=0.3, random_state=42
)

pipeline.fit(X_trn, y_trn)

print(X_train.shape, X_test.shape)
print(f"{pipeline.named_steps['prep'].transform(X_train).shape}")
print(f"{pipeline.named_steps['prep'].transform(X_test).shape}")

y_val_pred = pipeline.predict(X_val)

val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
val_mae  = mean_absolute_error(y_val, y_val_pred)
val_r2   = r2_score(y_val, y_val_pred)

print(f'Validation RMSE : {val_rmse:.4f}')
print(f'Validation MAE  : {val_mae:.4f}')
print(f'Validation R²   : {val_r2:.4f}')

(61609, 54) (41074, 54)
(61609, 66)
(41074, 66)
Validation RMSE : 19.4888
Validation MAE  : 15.7572
Validation R²   : 0.1865


In [37]:
test_preds =np.clip(pipeline.predict(X_test), 0, 100)
id_series = test_ids.squeeze().reset_index(drop=True)

submission = pd.DataFrame({
    'id': id_series,
    'target': test_preds
})

submission.to_csv('submission_linear.csv', index=False)

print('Saved: submission_linear.csv')


Saved: submission_linear.csv
